# 03 — SHAP Explanations (Group-Level)

Computes Mean Absolute SHAP importance aggregated at three levels of granularity
for every run defined in `manifesto_config.pkl`.

| Level | `Group_Variable` | `Group_Value` |
|---|---|---|
| Overall model | `Global` | `Global` |
| By actual-value decile | `Decile` | `D1` … `D10` |
| By segment (or any group_var) | e.g. `Segment` | e.g. `Platinum` |

**SHAP routing by model family**
| Family | Method | Notes |
|---|---|---|
| `Linear` | Exact analytical: `coef_j × (x_j − E[x_j])` | FE_LSDV / statsmodels |
| `Tree` | `shap.TreeExplainer` | XGBoost / LightGBM |
| `Neural` | `shap.GradientExplainer` + flat-sequence approx | LSTM / PyTorch |

**Output**: `Outputs/shap_importance.csv`  
**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils import load_model_artifacts, compute_shap, aggregate_shap

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Libraries loaded')

## 1. Load Manifesto

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']
group_vars      = manifesto['group_vars']

# DEV run lookup per model — used as SHAP background (E[x])
dev_lookup = {r['model']: r for r in dataset_configs if r['dataset'] == 'DEV'}

print(f'Runs scheduled : {len(dataset_configs)}')
print(f'Group vars     : {group_vars}')
print(f'Models with DEV background : {list(dev_lookup.keys())}')

## 2. SHAP Pipeline — All Runs

In [ ]:
MIN_GROUP_SAMPLES = 5
all_results       = []
artifacts_cache   = {}   # model_name -> (model_obj, features, aux)
background_cache  = {}   # model_name -> background_df

for run in dataset_configs:
    run_id       = run['run_id']
    model_name   = run['model']
    model_family = run['model_family']
    model_type   = run['model_type']
    run_gvars    = run['group_vars']

    print(f'\n[{run_id}]  family={model_family}  type={model_type}')

    # ── Load prediction file (always from manifest path) ──────────────────────
    df = pd.read_csv(run['path'])

    # ── Background: DEV predictions for same model ────────────────────────────
    if model_name not in background_cache:
        dev_run = dev_lookup.get(model_name)
        background_cache[model_name] = pd.read_csv(dev_run['path']) if dev_run else df
    background_df = background_cache[model_name]

    # ── Load model artifacts (cached per model) ───────────────────────────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts ...')
        model_obj, features, aux = load_model_artifacts(run)
        artifacts_cache[model_name] = (model_obj, features, aux)
        print(f'  Features : {len(features)}')
    else:
        model_obj, features, aux = artifacts_cache[model_name]

    # ── Compute SHAP ──────────────────────────────────────────────────────────
    shap_arr, eff_features = compute_shap(
        df, features, model_obj, model_family, aux, background_df,
        model_type=model_type,
    )
    print(f'  SHAP array : {shap_arr.shape}  eff_features={len(eff_features)}')

    # ── Aggregate at Global / Decile / group_vars ─────────────────────────────
    agg_df = aggregate_shap(
        shap_arr, eff_features, df, run, run_gvars,
        min_samples=MIN_GROUP_SAMPLES
    )
    all_results.append(agg_df)

    for gvar, grp in agg_df.groupby('Group_Variable'):
        n_buckets = grp['Group_Value'].nunique()
        print(f'  [{gvar}] — {n_buckets} bucket(s)')

print('\nSHAP pipeline complete.')

## 3. Save Output

In [ ]:
shap_df = pd.concat(all_results, ignore_index=True)

OUTPUT_COLS = [
    'Dataset', 'Model', 'Model_Domain', 'Model_Type',
    'Group_Variable', 'Group_Value', 'Samples',
    'Feature', 'Mean_Abs_SHAP',
]
shap_df = shap_df[[c for c in OUTPUT_COLS if c in shap_df.columns]]
shap_df = shap_df.sort_values(
    ['Dataset', 'Model', 'Group_Variable', 'Group_Value', 'Mean_Abs_SHAP'],
    ascending=[True, True, True, True, False]
).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, 'shap_importance.csv')
shap_df.to_csv(out_path, index=False)

n_global = (shap_df['Group_Variable'] == 'Global').sum()
n_decile = (shap_df['Group_Variable'] == 'Decile').sum()
n_group  = (~shap_df['Group_Variable'].isin(['Global', 'Decile'])).sum()
print(f'Saved: shap_importance.csv')
print(f'  Total rows   : {len(shap_df):,}')
print(f'  Global rows  : {n_global:,}')
print(f'  Decile rows  : {n_decile:,}')
print(f'  Group rows   : {n_group:,}')

## 4. Quick Views

In [ ]:
# Top 10 features globally per model (DEV only)
dev_global = shap_df[
    (shap_df['Dataset'] == 'DEV') & (shap_df['Group_Variable'] == 'Global')
]
print('── Top 10 Features by Mean |SHAP| — DEV Global ──────────────')
display(
    dev_global.groupby('Model')
    .apply(lambda x: x.nlargest(10, 'Mean_Abs_SHAP')[['Feature', 'Mean_Abs_SHAP', 'Samples']])
    .reset_index(level=0)
)

In [ ]:
# Segment-level top features (DEV only, first model)
dev_seg = shap_df[
    (shap_df['Dataset'] == 'DEV') & (shap_df['Group_Variable'] == 'Segment')
]
if not dev_seg.empty:
    print('── Top 5 Features per Segment — DEV ─────────────────────────')
    display(
        dev_seg.groupby(['Model', 'Group_Value'])
        .apply(lambda x: x.nlargest(5, 'Mean_Abs_SHAP')[['Feature', 'Mean_Abs_SHAP']])
        .reset_index(level=[0, 1])
    )